# Q, K and V

**One vector, three projections, three purposes.**

Every token entering an attention layer carries a single 2048-dimensional
vector — its hidden state. This notebook shows what happens when that one
vector hits the three weight matrices that transform it into Query, Key,
and Value.

No RoPE, no softmax, no attention — just the projection step.

Companion to [Episode 12](https://huggingface.co/blog/EXDai/qkv).

---

## Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"PyTorch {torch.__version__}")

In [ ]:
import os

MODEL_NAME = "Qwen/Qwen3.6-35B-A3B"

CACHE_DIR = os.path.expanduser("~/cache/hf/hub")
if os.path.isdir(CACHE_DIR):
    os.environ["HF_HOME"] = CACHE_DIR
    os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
    print(f"Using cache: {CACHE_DIR}")
else:
    print(f"Cache dir not found, using default")

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

lm = model.model
cfg = lm.config
print(f"Hidden size:   {cfg.hidden_size}")
print(f"Num layers:    {cfg.num_hidden_layers}")
print(f"Num Q heads:   {cfg.num_attention_heads}")
print(f"Num KV heads:  {cfg.num_key_value_heads}")
print(f"Head dim:      {cfg.head_dim}")

---

## Pick a Single Token

We feed the model a short sentence, then pick one token's hidden state
from the embedding layer — the vector as it enters the transformer for
the very first time.

In [ ]:
text = "The cat sat on the mat."

inputs = tok(text, return_tensors="pt").to(model.device)
token_ids = inputs["input_ids"][0]
tokens = [tok.decode(tid) for tid in token_ids]

print(f"Text:   {text}")
print(f"Tokens: {tokens}")
print(f"IDs:    {token_ids.tolist()}")

In [ ]:
# Grab the embedding for a specific token position
token_pos = 2  # "cat" in our sentence

with torch.no_grad():
    embeds = lm.embed_tokens(inputs["input_ids"])  # (1, seq, 2048)
    x = embeds[0, token_pos]  # (2048,)

print(f"Token:        '{tokens[token_pos]}'")
print(f"Vector shape: {x.shape}")
print(f"dtype:        {x.dtype}")
print(f"Norm (L2):    {x.norm().item():.4f}")
print(f"Min / Max:    {x.min().item():.4f} / {x.max().item():.4f}")

---

## The Input Vector

2048 numbers. That's all a token is to the model. Let's look at
what one of these vectors actually looks like.

In [ ]:
x_np = x.float().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. First 128 dimensions as bars
ax = axes[0]
ax.bar(range(128), x_np[:128], width=1, color='steelblue', alpha=0.8)
ax.set_xlabel("Dimension")
ax.set_ylabel("Value")
ax.set_title(f"First 128 of 2048 dimensions — token '{tokens[token_pos]}'")
ax.axhline(y=0, color='black', linewidth=0.5)

# 2. Histogram of all values
ax = axes[1]
ax.hist(x_np, bins=100, color='steelblue', alpha=0.8, edgecolor='white')
ax.set_xlabel("Value")
ax.set_ylabel("Count")
ax.set_title("Distribution of all 2048 values")
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)

# 3. Cumulative variance explained by dimensions
ax = axes[2]
sorted_vals = np.sort(np.abs(x_np))[::-1]
cumsum = np.cumsum(sorted_vals ** 2)
ax.plot(cumsum / cumsum[-1], color='steelblue')
ax.set_xlabel("Dimension rank (by |value|)")
ax.set_ylabel("Cumulative variance")
ax.set_title("How many dimensions carry the signal?")
ax.axhline(y=0.9, color='red', linestyle='--', alpha=0.5, label='90%')
ax.legend()

plt.tight_layout()
plt.show()

---

## The Weight Matrices

At the first full-attention layer, three learned matrices wait
to transform every incoming vector. Let's get them from the model.

In [ ]:
# Find the first full-attention layer
full_attn_layers = [i for i, layer in enumerate(lm.layers) if hasattr(layer, 'self_attn')]
target_layer = full_attn_layers[0]
attn = lm.layers[target_layer].self_attn

print(f"First full-attention layer: {target_layer}")
print(f"Module: {attn.__class__.__name__}")
print()

# Extract the weight matrices (no bias in Qwen3.5 attention)
W_Q = attn.q_proj.weight   # (8192, 2048) — includes gate
W_K = attn.k_proj.weight   # (512,  2048)
W_V = attn.v_proj.weight   # (512,  2048)

print(f"W_Q shape: {list(W_Q.shape)}  —  2048 → 8192  (16 heads × 256 × 2 for gate)")
print(f"W_K shape: {list(W_K.shape)}  —  2048 → 512   (2 heads × 256)")
print(f"W_V shape: {list(W_V.shape)}  —  2048 → 512   (2 heads × 256)")
print(f"\nTotal QKV parameters: {W_Q.numel() + W_K.numel() + W_V.numel():,}")

---

## The Shape of the Projection

A single 2048-dim vector hits three matrices simultaneously:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Visualize each weight matrix as a heatmap
matrices = [
    (W_Q, f"W_Q — {W_Q.shape[0]}×{W_Q.shape[1]}", "8192 out\n(16Q+gate) →"),
    (W_K, f"W_K — {W_K.shape[0]}×{W_K.shape[1]}", "512 out\n(2KV) →"),
    (W_V, f"W_V — {W_V.shape[0]}×{W_V.shape[1]}", "512 out\n(2KV) →"),
]

for ax, (W, title, ylabel) in zip(axes, matrices):
    # Show a subset for visual clarity (first 256 output dims, first 256 input dims)
    w_sub = W[:min(256, W.shape[0]), :256].detach().float().cpu().numpy()
    im = ax.imshow(w_sub, cmap="RdBu_r", aspect="auto", vmin=-0.5, vmax=0.5)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Input dimension (first 256 of 2048)")
    ax.set_ylabel(ylabel)
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle("Attention Weight Matrices — Learned Projections", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Compute Q, K, V

This is the actual operation. One matrix-vector multiply each:

$$\mathbf{q} = x \cdot W_Q^T \qquad \mathbf{k} = x \cdot W_K^T \qquad \mathbf{v} = x \cdot W_V^T$$

Same $x$, three different destinations.

In [ ]:
with torch.no_grad():
    # x is (2048,). W matrices are (out_features, 2048).
    # We need x as (2048,) @ (2048, out) but PyTorch Linear does W @ x + b
    # So we use the module directly
    q_full = attn.q_proj(x.unsqueeze(0)).squeeze(0)  # (8192,)
    k_full = attn.k_proj(x.unsqueeze(0)).squeeze(0)  # (512,)
    v_full = attn.v_proj(x.unsqueeze(0)).squeeze(0)  # (512,)

print(f"Q vector: {q_full.shape}  —  norm: {q_full.norm().item():.4f}")
print(f"K vector: {k_full.shape}  —  norm: {k_full.norm().item():.4f}")
print(f"V vector: {v_full.shape}  —  norm: {v_full.norm().item():.4f}")

---

## Reshape Into Heads

The flat vectors aren't the final form. Q splits into 16 query heads
(plus 16 gate channels). K and V split into just 2 KV heads each.

This is the asymmetry at the heart of GQA (Grouped Query Attention).

In [ ]:
num_q_heads = cfg.num_attention_heads       # 16
num_kv_heads = cfg.num_key_value_heads      # 2
head_dim = cfg.head_dim                     # 256

# Q: split into query + gate halves, then reshape each into heads
q_query, q_gate = torch.chunk(q_full, 2, dim=-1)  # (4096,) each
Q_heads = q_query.view(num_q_heads, head_dim)     # (16, 256)
gate_heads = q_gate.view(num_q_heads, head_dim)   # (16, 256)

# K and V: reshape into heads
K_heads = k_full.view(num_kv_heads, head_dim)     # (2, 256)
V_heads = v_full.view(num_kv_heads, head_dim)     # (2, 256)

print(f"Q (query only): {Q_heads.shape}  —  {num_q_heads} heads × {head_dim} dim")
print(f"Q (gate):        {gate_heads.shape}  —  same shape, separate purpose")
print(f"K:               {K_heads.shape}  —  {num_kv_heads} heads × {head_dim} dim")
print(f"V:               {V_heads.shape}  —  {num_kv_heads} heads × {head_dim} dim")
print(f"\nGQA ratio: {num_q_heads // num_kv_heads} Q heads per KV head")

---

## Head Norms: Who's "Loud"?

Each head's L2 norm tells us how strongly it's activated.
Heads with higher norms will produce larger attention scores.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Q head norms
q_norms = Q_heads.norm(dim=-1).float().cpu().numpy()
colors_q = plt.cm.Blues(np.linspace(0.4, 0.9, num_q_heads))
axes[0].bar(range(num_q_heads), q_norms, color=colors_q, edgecolor='white')
axes[0].set_xlabel("Head")
axes[0].set_ylabel("L2 Norm")
axes[0].set_title(f"Q Heads — {num_q_heads} heads")
axes[0].set_xticks(range(num_q_heads))

# K head norms
k_norms = K_heads.norm(dim=-1).float().cpu().numpy()
colors_kv = plt.cm.Oranges(np.linspace(0.5, 0.9, num_kv_heads))
axes[1].bar(range(num_kv_heads), k_norms, color=colors_kv, edgecolor='white')
axes[1].set_xlabel("Head")
axes[1].set_title(f"K Heads — {num_kv_heads} heads")
axes[1].set_xticks(range(num_kv_heads))

# V head norms
v_norms = V_heads.norm(dim=-1).float().cpu().numpy()
axes[2].bar(range(num_kv_heads), v_norms, color=colors_kv, edgecolor='white')
axes[2].set_xlabel("Head")
axes[2].set_title(f"V Heads — {num_kv_heads} heads")
axes[2].set_xticks(range(num_kv_heads))

fig.suptitle(f"Head Norms for token '{tokens[token_pos]}' — Before QK Norm", fontsize=13)
plt.tight_layout()
plt.show()

---

## Are Q, K, V Actually Different?

They all come from the same input vector. If the weight matrices
were identical, Q, K, and V would be identical. Let's verify that
the model has learned to project them into genuinely different spaces.

We'll use PCA to visualize the head vectors in 2D.

In [ ]:
# Stack all head vectors for PCA
all_vectors = torch.cat([
    Q_heads,       # (16, 256)
    K_heads,       # (2, 256)
    V_heads,       # (2, 256)
], dim=0).float().cpu().numpy()  # (20, 256)

labels = (
    [f'Q head {i}' for i in range(num_q_heads)] +
    [f'K head {i}' for i in range(num_kv_heads)] +
    [f'V head {i}' for i in range(num_kv_heads)]
)

pca = PCA(n_components=2)
coords = pca.fit_transform(all_vectors)

fig, ax = plt.subplots(figsize=(10, 8))

# Q heads
ax.scatter(coords[:16, 0], coords[:16, 1], c='steelblue', s=120, label=f'Q ({num_q_heads} heads)',
           edgecolors='white', linewidth=0.5, zorder=3)
for i in range(num_q_heads):
    ax.annotate(str(i), (coords[i, 0], coords[i, 1]), fontsize=8, ha='center', va='center')

# K heads
ax.scatter(coords[16:18, 0], coords[16:18, 1], c='darkorange', s=200, label=f'K ({num_kv_heads} heads)',
           marker='s', edgecolors='white', linewidth=0.5, zorder=4)

# V heads
ax.scatter(coords[18:20, 0], coords[18:20, 1], c='crimson', s=200, label=f'V ({num_kv_heads} heads)',
           marker='^', edgecolors='white', linewidth=0.5, zorder=4)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title(f"Q, K, V Head Vectors in PCA Space — Token '{tokens[token_pos]}'")
ax.legend(loc='upper right')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")

---

## What Makes Q Heads Different From Each Other?

16 Q heads all came from the same input. Are they redundant copies
or genuinely different? Cosine similarity tells us.

In [ ]:
# Cosine similarity between all Q heads
Q_normed = Q_heads.float() / Q_heads.float().norm(dim=-1, keepdim=True)
cos_sim = (Q_normed @ Q_normed.T).cpu().numpy()  # (16, 16)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

im = ax1.imshow(cos_sim, cmap="RdBu_r", vmin=-1, vmax=1, aspect='auto')
ax1.set_xticks(range(num_q_heads))
ax1.set_yticks(range(num_q_heads))
ax1.set_xlabel("Head")
ax1.set_ylabel("Head")
ax1.set_title("Cosine Similarity — All 16 Q Heads")
plt.colorbar(im, ax=ax1, label="cos θ")

# For each head, show max similarity to any *other* head
off_diag = cos_sim.copy()
np.fill_diagonal(off_diag, -1)
max_other = off_diag.max(axis=1)

ax2.bar(range(num_q_heads), max_other, color=plt.cm.Blues(np.linspace(0.3, 0.8, num_q_heads)), edgecolor='white')
ax2.set_xlabel("Head")
ax2.set_ylabel("Max cos sim with any other head")
ax2.set_title("How Distinct Is Each Q Head?")
ax2.set_xticks(range(num_q_heads))
ax2.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax2.set_ylim(-0.2, 0.6)

fig.suptitle("Q Head Diversity", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## The GQA Asymmetry

16 Q heads. 2 KV heads. Each KV head serves 8 Q heads.

How different can 8 Q heads be if they'll all query the same K, V?
Let's look at the Q heads grouped by which KV head they pair with.

In [ ]:
n_rep = num_q_heads // num_kv_heads  # 8 Q heads per KV head

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for kv_idx in range(num_kv_heads):
    q_start = kv_idx * n_rep
    q_end = q_start + n_rep

    # Q heads for this KV head
    q_norms_group = q_norms[q_start:q_end]
    colors_group = plt.cm.Blues(np.linspace(0.3, 0.9, n_rep))

    # Plot on each subplot
    ax = axes[kv_idx]
    x_pos = np.arange(n_rep)
    ax.bar(x_pos, q_norms_group, color=colors_group, edgecolor='white')
    ax.set_xlabel(f"Q head ({q_start}–{q_end - 1})")
    ax.set_ylabel("L2 Norm")
    ax.set_title(f"Q heads sharing KV head {kv_idx}")
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(i) for i in range(q_start, q_end)])

    # Add K and V norm as horizontal lines
    ax.axhline(y=k_norms[kv_idx], color='darkorange', linestyle='--', linewidth=2, label=f'K head {kv_idx}')
    ax.axhline(y=v_norms[kv_idx], color='crimson', linestyle=':', linewidth=2, label=f'V head {kv_idx}')
    ax.legend(fontsize=9)

fig.suptitle(f"GQA Groups — {n_rep} Q heads share each KV head", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## Cross-QKV Similarity

Q heads query K heads. V heads are the payload.
How aligned are Q heads with the K heads they'll search against?
And are K and V similar to each other?

Recall: K and V come from the *same* input token (just like Q),
but through different weight matrices. If $W_K \approx W_V$,
then K and V would be nearly identical — the model would just
be copying itself. Let's check.

In [ ]:
# Cosine similarity: Q heads vs K heads, and K vs V
Q_flat = Q_heads.float()     # (16, 256)
K_flat = K_heads.float()     # (2,   256)
V_flat = V_heads.float()     # (2,   256)

Q_n = Q_flat / Q_flat.norm(dim=-1, keepdim=True)
K_n = K_flat / K_flat.norm(dim=-1, keepdim=True)
V_n = V_flat / V_flat.norm(dim=-1, keepdim=True)

# Q vs K: (16, 2)
qk_sim = (Q_n @ K_n.T).cpu().numpy()

# K vs V: (2, 2)
kv_sim = (K_n @ V_n.T).cpu().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

im1 = ax1.imshow(qk_sim, cmap="RdBu_r", vmin=-1, vmax=1, aspect='auto')
ax1.set_xlabel("K head")
ax1.set_ylabel("Q head")
ax1.set_xticks(range(num_kv_heads))
ax1.set_yticks(range(num_q_heads))
ax1.set_title(f"cos(Q, K) — 16 Q heads × 2 K heads")
plt.colorbar(im1, ax=ax1, label="cos θ")

im2 = ax2.imshow(kv_sim, cmap="RdBu_r", vmin=-1, vmax=1, aspect='auto')
ax2.set_xlabel("V head")
ax2.set_ylabel("K head")
ax2.set_xticks(range(num_kv_heads))
ax2.set_yticks(range(num_kv_heads))
ax2.set_title(f"cos(K, V) — 2 K heads × 2 V heads")
plt.colorbar(im2, ax=ax2, label="cos θ")

fig.suptitle("Cross-Component Similarities", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Mean |cos(Q, K)|:  {np.abs(qk_sim).mean():.4f}")
print(f"Mean |cos(K, V)|:  {np.abs(kv_sim).mean():.4f}")
print(f"K head 0 vs K head 1 cos: {cos_sim_kk:.4f}" if (cos_sim_kk := (K_n[0] @ K_n[1]).item()) else "")

---

## The Gate: Q's Hidden Passenger

Q's output is 8192-dim because it carries an extra payload:
a *gate* signal for each head and each dimension. The gate is
projected through the same $W_Q$ matrix (doubled width) for
efficiency, then split off.

The gate controls how much of the attention output to trust.
Gate → 0 silences the head. Gate → 1 lets it through.

In [ ]:
# Gate values per head (mean across the 256 dimensions)
gate_mean = gate_heads.float().mean(dim=-1).cpu().numpy()  # (16,)
gate_std = gate_heads.float().std(dim=-1).cpu().numpy()    # (16,)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Mean gate per head (pre-sigmoid)
ax1.bar(range(num_q_heads), gate_mean, color='steelblue', edgecolor='white')
ax1.set_xlabel("Head")
ax1.set_ylabel("Mean gate value (pre-sigmoid)")
ax1.set_title("Mean Gate Activation per Q Head")
ax1.set_xticks(range(num_q_heads))
ax1.axhline(y=0, color='black', linewidth=0.5)

# Show the actual gate values for one head
head_show = 0
gate_head0 = gate_heads[head_show].float().cpu().numpy()
ax2.bar(range(64), gate_head0[:64], width=1, color='teal', alpha=0.8)
ax2.set_xlabel(f"Dimension (first 64 of 256)")
ax2.set_ylabel("Gate value (pre-sigmoid)")
ax2.set_title(f"Gate Vector — Head {head_show}")
ax2.axhline(y=0, color='black', linewidth=0.5)

fig.suptitle(f"Gate — Attention's Volume Knob", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Gate mean across all heads: {gate_mean.mean():.4f}")
print(f"Gate std across all heads:  {gate_std.mean():.4f}")
print("\n(Values are pre-sigmoid. After sigmoid, they'd be squashed to [0, 1].)")
print("The sigmoid is applied later, during the attention computation, not here.")

---

## What Do the Weight Matrices Learn?

The weight matrices themselves have structure. Let's look at
their singular values — how many "effective directions" does
each projection use?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (W, name, color) in zip(axes, [
    (W_Q, "W_Q", "steelblue"),
    (W_K, "W_K", "darkorange"),
    (W_V, "W_V", "crimson"),
]):
    U, S, Vh = torch.linalg.svd(W.float())
    S_norm = S / S.max()
    ax.plot(S_norm.cpu().numpy(), color=color, linewidth=1.5)
    ax.set_xlabel("Singular value index")
    ax.set_ylabel("Normalized singular value")
    ax.set_title(f"{name} — {W.shape[0]}×{W.shape[1]}")
    ax.axhline(y=0.1, color='gray', linestyle='--', alpha=0.5)
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

fig.suptitle("Singular Value Spectra of Attention Weight Matrices", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## Different Tokens, Different Q/K/V

Every token in the sentence produces its own Q, K, V.
Let's compare the Q heads across all tokens — does "cat"
ask different questions than "mat"?

In [ ]:
# Compute Q heads for all tokens
all_Q_norms = []
all_labels = []

with torch.no_grad():
    for pos in range(len(tokens)):
        x_pos = embeds[0, pos]
        q_full_tok = attn.q_proj(x_pos.unsqueeze(0)).squeeze(0)
        q_query_tok, _ = torch.chunk(q_full_tok, 2, dim=-1)
        Q_tok = q_query_tok.view(num_q_heads, head_dim)
        all_Q_norms.append(Q_tok.norm(dim=-1).float().cpu().numpy())
        all_labels.append(tokens[pos])

all_Q_norms = np.array(all_Q_norms)  # (seq, 16)

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(tokens))
width = 0.8 / num_q_heads
for h in range(num_q_heads):
    offset = (h - num_q_heads / 2) * width
    ax.bar(x + offset, all_Q_norms[:, h], width=width * 0.9,
           color=plt.cm.Blues(0.3 + 0.7 * h / num_q_heads), alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(tokens, fontsize=10)
ax.set_xlabel("Token")
ax.set_ylabel("L2 Norm per head")
ax.set_title(f"Q Head Norms — All Tokens, All Heads")
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

# Which heads vary most across tokens?
head_variance = all_Q_norms.var(axis=0)
print(f"\nHead norm variance across tokens:")
for h in np.argsort(head_variance)[::-1][:5]:
    print(f"  Head {h}: σ²={head_variance[h]:.4f}")

---

## The Cost of Projection

The Q/K/V projection is a massive matrix multiplication.
Let's count the FLOPs.

In [ ]:
# One token, one layer: Q + K + V projections
flops_per_token = (
    2 * W_Q.shape[0] * W_Q.shape[1] +   # Q: 2 × 8192 × 2048
    2 * W_K.shape[0] * W_K.shape[1] +   # K: 2 × 512  × 2048
    2 * W_V.shape[0] * W_V.shape[1]     # V: 2 × 512  × 2048
)
print(f"FLOPs per token per layer: {flops_per_token:,}")
print(f"  Q: {2 * W_Q.shape[0] * W_Q.shape[1]:,} FLOPs ({(2 * W_Q.shape[0] * W_Q.shape[1]) / flops_per_token * 100:.0f}%)")
print(f"  K: {2 * W_K.shape[0] * W_K.shape[1]:,} FLOPs ({(2 * W_K.shape[0] * W_K.shape[1]) / flops_per_token * 100:.0f}%)")
print(f"  V: {2 * W_V.shape[0] * W_V.shape[1]:,} FLOPs ({(2 * W_V.shape[0] * W_V.shape[1]) / flops_per_token * 100:.0f}%)")
print(f"\nFor a {len(tokens)}-token sequence: {flops_per_token * len(tokens):,} FLOPs")
print(f"For all {len(full_attn_layers)} full-attention layers: {flops_per_token * len(tokens) * len(full_attn_layers):,} FLOPs")

---

## What We Saw

One 2048-dim hidden state. Three learned weight matrices.
Three different output vectors, reshaped into heads:

| Component | Heads | Dim each | Purpose |
|-----------|-------|----------|---------|
| **Q** (query) | 16 | 256 | "What am I looking for?" |
| **Gate** | 16 | 256 | "How much do I trust attention output?" |
| **K** (key) | 2 | 256 | "Here's how to find me" |
| **V** (value) | 2 | 256 | "Here's my content if you attend to me" |

Key observations:

- **Projection is all there is to it.** No activation functions, no
  nonlinearities — just matrix multiplication. The "intelligence" is
  in the learned weights.
- **Q, K, V occupy different subspaces.** PCA shows they cluster
  separately — the weight matrices have learned genuinely different
  projections from the same input.
- **Q heads are diverse but correlated.** The 16 heads aren't redundant
  copies; each projects the input differently.
- **GQA asymmetry is stark.** 16 Q heads, only 2 KV heads. The burden
  of "different questions" is on Q; K and V are lean.
- **The gate is a separate signal** riding on Q's projection.
  It will modulate attention output later — but it's born here,
  from the same $x \cdot W_Q$.

**Next:** QK Norm + RoPE rotate these vectors before they meet
in the attention computation (Episode 13).